## FRAP analyses - interpretation

___

In [ ]:
import sys; from pathlib import Path
src_dir = next(parent / 'src' for parent in Path().absolute().parents if (parent / 'src').is_dir())
sys.path.extend([str(src_dir)])
from microlive.pipelines.pipeline_FRAP import *
from imports import * ; current_dir = Path().resolve()
import tifffile
plt.rcParams.update(plt.rcParamsDefault)
from frap_plotting_module import *


In [ ]:
path_main_folder = Path('/Users/nzlab-la/Library/CloudStorage/OneDrive-TheUniversityofColoradoDenver/General - Zhao (NZ) Lab/Zhao lab shared folder/Our papers/Anti-Utag-frankenbody paper/Fig. 4 FRAP')
results_folder = path_main_folder.joinpath('publication_images') 
results_folder.mkdir(exist_ok=True)

## Parameters
___

In [ ]:
frap_time = 10
stable_FRAP_channel = 1 
FRAP_channel_to_quantify = 0
radius_roi_size_px=14
save_individual_dataframes = False
fit_model_considering_immobile_fraction = False 
# adjusting the frame rate for the FRAP images
starting_changing_frame=40
step_size_increase=5
length_kymograph_line = 50
cmap_list = [green_colormap, magenta_colormap]
list_selected_frame_values_real_time =[0, 10, 50, 100, 300, 500] 
list_selected_frame_values_real_time_after_downsampling =[0, 2, 10, 100, 300, 500] 
x_title_list = ['', 'FRAP', ' 50s', '100s', '300s', '500s']
list_datasets = ['suntag']
min_size_image = 230
scalebar_size = 0

save_tif = True

### Main functions
____

### Plotting functions
All plotting functions are imported from `frap_plotting_module.py`.

### Running
___

In [ ]:
for index_folder in range (len(list_datasets)):
    list_mean_roi_frap_normalized =[]
    data_folder_path, selected_image_name, list_axis_limits, y_label_list,plot_name= get_data_folder_path(dataset=list_datasets[index_folder], path_main_folder=path_main_folder)
    # load the images 
    list_images, list_names, pixel_xy_um, voxel_z_um, channel_names, number_color_channels,list_time_intervals, bit_depth,_,_,_ = mi.ReadLif(data_folder_path,show_metadata=False,save_tif=False,save_png=False,format='TZYXC').read()
    # concatenating images pre and post FRAP time
    list_concatenated_images, list_names_concatenated_images,list_time_concatenated = concatenate_images(list_images, list_names,False,list_time_intervals )
    for i in range(len(list_names_concatenated_images)):
        if selected_image_name == list_names_concatenated_images[i]:
            # This function segments the nuclei and background of the image and return the image in the correct format for the FRAP analysis
            image_TZXYC, image_TZXYC_masked, image_TXY, masks_TXY, pseudo_cytosol_masks_TXY, background_mask,frame_values = create_image_arrays(list_concatenated_images, selected_image=i,
                                                                                                                    FRAP_channel_to_quantify=FRAP_channel_to_quantify,
                                                                                                                    pretrained_model_segmentation=None,
                                                                                                                    frap_time=frap_time, starting_changing_frame=starting_changing_frame, 
                                                                                                                    step_size_increase=step_size_increase, min_diameter=radius_roi_size_px*2)
            # save image_TZXYC as tif
            if save_tif:
                image_TZCYX = np.transpose(image_TZXYC, (0, 1, 4, 3, 2))  # T,Z,X,Y,C -> T,Z,C,Y,X
                # Correct transposition from TZXYC to TZCYX for ImageJ compatibility
                name_tif = results_folder.joinpath(list_datasets[index_folder] +f"_{selected_image_name.replace(' ', '_')}.tif" )
                tifffile.imwrite(
                    name_tif, 
                    image_TZCYX,  # Use transposed array
                    imagej=True,
                    ome=False,
                    resolution=(pixel_xy_um, pixel_xy_um),
                    resolutionunit='MICROMETER',
                    metadata={
                        'axes': 'TZCYX',  
                        'PhysicalSizeX': pixel_xy_um,
                        'PhysicalSizeY': pixel_xy_um, 
                        'PhysicalSizeZ': voxel_z_um,
                        'PhysicalSizeXUnit': 'um',
                        'PhysicalSizeYUnit': 'um',
                        'PhysicalSizeZUnit': 'um'
                    }
                )

            # Convert the used selected frames to the index of the frames
            list_selected_frames = (  [find_nearest(frame_values, time) for time in list_selected_frame_values_real_time])
            mask_intensity_nucleus, mask_intensity_background, mask_intensity_pseudo_cytosol = calculate_mask_and_background_intensity(image_TXY, masks_TXY,background_mask,pseudo_cytosol_masks_TXY)
            # this section detects the roi in the image.
            mean_roi_frap,mean_roi_frap_normalized, coordinates_roi, df_selected_trajectory = find_frap_roi(image_TZXYC_masked,image_TZXYC, masks_TXY, frap_time,
                                                                                                            min_diameter=radius_roi_size_px*2,
                                                                                                            FRAP_channel_to_quantify=FRAP_channel_to_quantify,
                                                                                                            stable_FRAP_channel=stable_FRAP_channel,
                                                                                                            show_binary_plot=False,
                                                                                                            mask_intensity_background=mask_intensity_background,
                                                                                                            use_frap_time_for_roi_detection= True,)
            list_mean_roi_frap_normalized.append(mean_roi_frap_normalized)
            
            # Plotting 
            plot_images_frap_all_channels_representative(image_TZXYC, results_folder=results_folder, pixel_xy_um=pixel_xy_um, scalebar_size=scalebar_size, list_selected_frames=list_selected_frames, coordinates_roi=coordinates_roi, radius_roi_size_px=radius_roi_size_px, cmap_list =cmap_list,list_axis_limits=list_axis_limits,
                                            y_label_list=y_label_list,list_selected_frame_values_real_time=list_selected_frame_values_real_time,x_title_list=x_title_list,plot_name=plot_name,masks_TXY=masks_TXY,min_size_image=min_size_image) 
            
            path_kymograph = plot_kymograph_downsampled(image_TZXYC, coordinates_roi, list_selected_frames, x_title_list, list_selected_frame_values_real_time=list_selected_frame_values_real_time, frame_values=frame_values, results_folder=results_folder, length_kymograph_line=length_kymograph_line, cmap_list=cmap_list, plot_vertical_lines=True, plot_name=plot_name)
            
            path_merged_image = plot_merged_image(image_TZXYC, coordinates_roi, length_kymograph_line, cmap_list_imagej=cmap_list, results_folder=results_folder, pixel_xy_um=pixel_xy_um, list_axis_limits=list_axis_limits, plot_name=plot_name, masks_TXY=masks_TXY, min_size_image=min_size_image)

            # save avi video
            avi_name = results_folder.joinpath('FRAP_anti-'+plot_name+'.avi')
            save_video_as_avi(image_TZXYC,avi_name, frame_values, list_axis_limits,cmap_list,y_label_list, fps=10,pixel_xy_um=pixel_xy_um, scalebar_size=scalebar_size)

            compose_pngs( path_merged_image, path_kymograph, results_folder.joinpath('FRAP_anti-'+plot_name+'.png'), target_height=500)